In [1]:
%reload_ext autoreload
%autoreload 2 --print

In [2]:
import src.config as config

In [3]:
# connect to the feature store and feature group where we want to save the data
import hopsworks

# connect to the project
project= hopsworks.login(
    project= config.HOPSWORKS_PROJECT_NAME,
    api_key_value= config.HOPSWORKS_API_KEY
)

# connect to the feature store
feature_store= project.get_feature_store()

# connect to the feature group 
feature_group= feature_store.get_or_create_feature_group(
    name= config.FEATURE_GROUP_NAME,
    version= config.FEATURE_GROUP_VERSION,
)

2026-11-13 16:14:55,081 INFO: Initializing external client
2026-11-13 16:14:55,083 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/55417


In [5]:
# to read the data from the feature store, we define a feature view
# create a feature view (if not exists already)
# this feature view only uses one feature group, so the query is trivial

try:
    feature_store.create_feature_view(
        name= config.FEATURE_VIEW_NAME,
        version= config.FEATURE_VIEW_VERSION,
        query= feature_group.select_all()
    )
except:
    print("Feature view already exists")

# get the feature view
feature_view= feature_store.get_feature_view(
    name= config.FEATURE_VIEW_NAME,
    version= config.FEATURE_VIEW_VERSION
)

Reloading 'src.config'.
Feature view created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/55417/fs/43078/fv/time_series_hourly_feature_view/version/1


In [6]:
# getting the training data from feature store using the feature view

ts_data, _ = feature_view.training_data(
    description= "Time-series hourly taxi rides",
)

2026-11-13 16:28:11,308 INFO: Python Engine initialized.


2026-11-13 16:28:52,667 INFO: Computing insert statistics


Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (36.81s) 


In [10]:
ts_data.sort_values(by= ["pickup_location_id","pickup_hour"], inplace= True)
ts_data

,pickup_hour,rides,pickup_location_id
2816897,2022-01-01 01:00:00+00:00,0,1
5743644,2022-01-01 02:00:00+00:00,0,1
4143970,2022-01-01 05:00:00+00:00,1,1
638882,2022-01-01 09:00:00+00:00,0,1
644936,2022-01-01 09:00:00+00:00,0,1
...,...,...,...
3174789,2026-10-30 19:00:00+00:00,3,265
3175049,2026-10-30 20:00:00+00:00,3,265
3175309,2026-10-30 21:00:00+00:00,2,265
3175569,2026-10-30 22:00:00+00:00,3,265


In [13]:
# transform the training data into features and targets
from src.data import transform_ts_data_into_features_and_target

features, targets= transform_ts_data_into_features_and_target(
    ts_data,
    input_seq_len= 24*28,
    step_size= 23
)

features_and_target= features.copy()
features_and_target["target_rides_next_hour"]= targets

print(f"{features_and_target.shape= }")


100%|██████████| 263/263 [05:22<00:00,  1.23s/it]


features_and_target.shape= (258916, 675)


In [ ]:
# splitting the data into train and test sets
import pandas as pd
from src.data_split import train_test_split

# training data -> from January 2022 up until one month ago
# test data -> last one month
 
cutoff_date = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=28)

print(f"{cutoff_date= }")


X_train, y_train, X_test, y_test= train_test_split(
    features_and_target,
    cutoff_date,
    target_column_name= "target_rides_next_hour"
)

print(f"{X_train.shape= }")
print(f"{y_train.shape= }")
print(f"{X_test.shape= }")
print(f"{y_test.shape= }")



cutoff_date= Timestamp('2026-10-16 06:25:42.354056+0000', tz='UTC')
X_train.shape= (254928, 674)
y_train.shape= (254928,)
X_test.shape= (3988, 674)
y_test.shape= (3988,)


In [ ]:
# training the model (lightbgm + feature engineering + hyperparameter tuning)

import numpy as np
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error
import optuna

from src.model import get_pipeline

def objective(trial: optuna.trial.Trial) -> float:
    """
    Given a set of hyperparameters, it trains a model and computes
    an average validation error based on a TimeSeriesSplit.
    """
    # select hyperparameters
    hyperparams ={
        "metric": "mae",
        "verbose": -1,
        "num_leaves": trial.suggest_int("num_leaves", 2, 256),
        "feature_fraction": trial.suggest_float("feature_fraction:",0.2, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.2, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples",3 ,100)
    }

    tss= TimeSeriesSplit(n_splits=4)
    scores= []
    for train_index, val_index in tss.split(X_train):

        # split the data into training and validation sets
        X_train_, X_val_= X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_, y_val_= y_train.iloc[train_index], y_train.iloc[val_index]

        # train the model
        pipeline= get_pipeline(**hyperparams)
        pipeline.fit(X_train_, y_train_)

        # evaluate the model
        y_pred= pipeline.predict(X_val_)
        mae= mean_absolute_error(y_val_, y_pred)

        scores.append(mae)

        # Return the mean score
        return np.array(scores).mean()

In [25]:
# creata an Optuna study and optimize the objective function
study= optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=5)

[I 2026-11-13 17:30:21,444] A new study created in memory with name: no-name-0c617aa5-25e1-40a3-baa5-57951b838979
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
[I 2026-11-13 17:31:03,947] Trial 0 finished with value: 4.291051361932781 and parameters: {'num_leaves': 217, 'feature_fraction:': 0.5257316105013838, 'bagging_fraction': 0.41686497187775573, 'min_child_samples': 37}. Best is trial 0 with value: 4.291051361932781.
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,

In [27]:
# extract the best parameters found
best_params= study.best_trial.params
print(f"{best_params= }")

best_params= {'num_leaves': 33, 'feature_fraction:': 0.8366035116485342, 'bagging_fraction': 0.5382628470221267, 'min_child_samples': 44}


In [28]:
# retrain the model on the entire data with the best params found
pipeline= get_pipeline(**best_params)
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('functiontransformer', ...), ('temporalfeatureengineer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](674,)","['rides_previous_672_hour','rides_previous_671_hour', 'rides_previous_670_hour',...,'rides_previous_1_hour','pickup_hour', 'location_id']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,674
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function ave...00196F5422020>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False


In [31]:
predictions= pipeline.predict(X_test)
test_mae= mean_absolute_error(y_test, predictions)
print(f"{test_mae= :.4f}")

test_mae= 5.3251


In [ ]:
# first saving the model on local storage
import joblib
from src.paths import MODELS_DIR

joblib.dump(pipeline, MODELS_DIR/"model.pkl")

Reloading 'src.paths'.


['D:\\projects\\nyc_taxi_demand_predictor\\models\\model.pkl']

In [34]:
# before pushing the model to Hopsworks model registry, we define a Hopsworks model schema
from hsml.schema import Schema
from hsml.model_schema import ModelSchema

input_schema= Schema(X_train)
output_schema= Schema(y_train)
model_schema= ModelSchema(input_schema= input_schema, output_schema= output_schema)

In [37]:
# now we push the model into model registry

model_registry= project.get_model_registry()

model= model_registry.sklearn.create_model(
    name= "taxi_demand_predictor_next_hour",
    metrics= {"test_mae": test_mae},
    description= "LightGBM regressor + feature engineering + hyperparameter tuning",
    input_example= X_train.sample(),
    model_schema= model_schema
)

model.save(str(MODELS_DIR/'model.pkl'))

  0%|          | 0/6 [00:00<?, ?it/s]

Moving model files from 'D:\projects\nyc_taxi_demand_predictor\models\model.pkl' to the model registry... This is the default behavior. Set keep_original_files=True to copy files instead.


Uploading D:\projects\nyc_taxi_demand_predictor\models\model.pkl: 0.000%|          | 0/357580 elapsed<00:00 re…

Uploading C:\Users\NIMAMA~1\AppData\Local\Temp\tmp7s4q73xv\input_example.json: 0.000%|          | 0/4000 elaps…

Uploading C:\Users\NIMAMA~1\AppData\Local\Temp\tmp7s4q73xv\model_schema.json: 0.000%|          | 0/60842 elaps…

Model created, explore it at https://eu-west.cloud.hopsworks.ai:443/p/55417/models/taxi_demand_predictor_next_hour/1


Model(name: 'taxi_demand_predictor_next_hour', version: 1)